In [ ]:
# ══════════════════════════════════════════════════════════════
# PYSPARK + MONGODB — Google Colab
# Lectura, procesamiento y escritura entre Spark y MongoDB
# ══════════════════════════════════════════════════════════════

# Instalamos las librerías necesarias
!pip install pyspark pymongo mongomock -q

import mongomock
import pymongo
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

# ──────────────────────────────────────────────
# 1. CONEXIÓN A MONGODB Y CARGA DE DATOS
# ──────────────────────────────────────────────
print("=" * 60)
print("1. CARGA DE DATOS EN MONGODB")
print("=" * 60)

print("""
En este ejemplo simulamos el flujo real de trabajo:
  1. MongoDB almacena datos operacionales (documentos JSON)
  2. PySpark los lee, los procesa de forma distribuida
  3. Los resultados se escriben de vuelta en MongoDB

En producción real usaríamos el conector oficial:
  spark.read.format("mongo").option("uri", "mongodb://...").load()

En Colab simulamos MongoDB con mongomock para no necesitar servidor.
""")

# Conectamos a MongoDB (simulado con mongomock)
client = mongomock.MongoClient()
db = client["ecommerce"]

# Insertamos datos operacionales — simulan lo que llegaría
# desde un sistema de ventas en tiempo real
ventas_docs = [
    {"id_pedido": "P001", "cliente": "Ana García",    "ciudad": "Madrid",    "producto": "Laptop Pro",        "categoria": "Electrónica", "precio": 1299.99, "cantidad": 2, "fecha": "2023-01-15", "valoracion": 5},
    {"id_pedido": "P002", "cliente": "Carlos López",  "ciudad": "Barcelona", "producto": "Monitor 4K",        "categoria": "Electrónica", "precio": 549.99,  "cantidad": 1, "fecha": "2023-01-20", "valoracion": 4},
    {"id_pedido": "P003", "cliente": "María Ruiz",    "ciudad": "Valencia",  "producto": "Silla Ergonómica",  "categoria": "Mobiliario",  "precio": 299.99,  "cantidad": 3, "fecha": "2023-02-05", "valoracion": 5},
    {"id_pedido": "P004", "cliente": "Ana García",    "ciudad": "Madrid",    "producto": "Teclado Mecánico",  "categoria": "Electrónica", "precio": 129.99,  "cantidad": 1, "fecha": "2023-02-10", "valoracion": 3},
    {"id_pedido": "P005", "cliente": "Pedro Sánchez", "ciudad": "Sevilla",   "producto": "Laptop Pro",        "categoria": "Electrónica", "precio": 1299.99, "cantidad": 1, "fecha": "2023-03-01", "valoracion": 5},
    {"id_pedido": "P006", "cliente": "Laura Martín",  "ciudad": "Madrid",    "producto": "Mesa de Oficina",   "categoria": "Mobiliario",  "precio": 450.00,  "cantidad": 2, "fecha": "2023-03-15", "valoracion": 4},
    {"id_pedido": "P007", "cliente": "Carlos López",  "ciudad": "Barcelona", "producto": "Auriculares BT",    "categoria": "Electrónica", "precio": 129.99,  "cantidad": 2, "fecha": "2023-04-01", "valoracion": 4},
    {"id_pedido": "P008", "cliente": "María Ruiz",    "ciudad": "Valencia",  "producto": "Monitor 4K",        "categoria": "Electrónica", "precio": 549.99,  "cantidad": 1, "fecha": "2023-04-10", "valoracion": 5},
    {"id_pedido": "P009", "cliente": "Pedro Sánchez", "ciudad": "Sevilla",   "producto": "Silla Ergonómica",  "categoria": "Mobiliario",  "precio": 299.99,  "cantidad": 1, "fecha": "2023-05-05", "valoracion": 3},
    {"id_pedido": "P010", "cliente": "Ana García",    "ciudad": "Madrid",    "producto": "Webcam HD",         "categoria": "Electrónica", "precio": 89.99,   "cantidad": 1, "fecha": "2023-05-20", "valoracion": 4},
    {"id_pedido": "P011", "cliente": "Laura Martín",  "ciudad": "Madrid",    "producto": "Laptop Pro",        "categoria": "Electrónica", "precio": 1299.99, "cantidad": 1, "fecha": "2023-06-01", "valoracion": 5},
    {"id_pedido": "P012", "cliente": "Carlos López",  "ciudad": "Barcelona", "producto": "Mesa de Oficina",   "categoria": "Mobiliario",  "precio": 450.00,  "cantidad": 1, "fecha": "2023-06-15", "valoracion": 4},
    {"id_pedido": "P013", "cliente": "Pedro Sánchez", "ciudad": "Sevilla",   "producto": "Teclado Mecánico",  "categoria": "Electrónica", "precio": 129.99,  "cantidad": 3, "fecha": "2023-07-01", "valoracion": 5},
    {"id_pedido": "P014", "cliente": "María Ruiz",    "ciudad": "Valencia",  "producto": "Webcam HD",         "categoria": "Electrónica", "precio": 89.99,   "cantidad": 2, "fecha": "2023-07-20", "valoracion": 3},
    {"id_pedido": "P015", "cliente": "Ana García",    "ciudad": "Madrid",    "producto": "Auriculares BT",    "categoria": "Electrónica", "precio": 129.99,  "cantidad": 4, "fecha": "2023-08-05", "valoracion": 4},
]

db["ventas"].insert_many(ventas_docs)
print(f"✔ {db['ventas'].count_documents({})} documentos insertados en MongoDB")

# También insertamos una colección de productos con info adicional
productos_docs = [
    {"producto": "Laptop Pro",       "categoria": "Electrónica", "proveedor": "TechCorp",   "coste": 800.00,  "stock": 50},
    {"producto": "Monitor 4K",       "categoria": "Electrónica", "proveedor": "ScreenCo",   "coste": 300.00,  "stock": 30},
    {"producto": "Silla Ergonómica", "categoria": "Mobiliario",  "proveedor": "FurniCo",    "coste": 150.00,  "stock": 80},
    {"producto": "Teclado Mecánico", "categoria": "Electrónica", "proveedor": "TechCorp",   "coste": 60.00,   "stock": 120},
    {"producto": "Mesa de Oficina",  "categoria": "Mobiliario",  "proveedor": "FurniCo",    "coste": 200.00,  "stock": 25},
    {"producto": "Auriculares BT",   "categoria": "Electrónica", "proveedor": "SoundTech",  "coste": 55.00,   "stock": 90},
    {"producto": "Webcam HD",        "categoria": "Electrónica", "proveedor": "VisionCo",   "coste": 35.00,   "stock": 60},
]
db["productos"].insert_many(productos_docs)
print(f"✔ {db['productos'].count_documents({})} productos insertados en MongoDB")


# ──────────────────────────────────────────────
# 2. LEER MONGODB → PYSPARK DATAFRAME
# ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("2. MONGODB → PYSPARK DATAFRAME")
print("=" * 60)

print("""
El conector oficial MongoDB-Spark convierte documentos BSON
a filas de un DataFrame automáticamente.
Aquí lo simulamos leyendo desde mongomock y creando el DataFrame.
En producción sería: spark.read.format("mongo").load()
""")

# Arrancamos Spark
spark = SparkSession.builder \
    .appName("PySpark + MongoDB") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Leemos los documentos de MongoDB y los convertimos a DataFrame
# En producción: spark.read.format("mongo").option("uri","mongodb://...").load()
ventas_list = list(db["ventas"].find({}, {"_id": 0}))
productos_list = list(db["productos"].find({}, {"_id": 0}))

# Definimos el esquema explícitamente para mayor control
schema_ventas = StructType([
    StructField("id_pedido",  StringType(),  False),
    StructField("cliente",    StringType(),  True),
    StructField("ciudad",     StringType(),  True),
    StructField("producto",   StringType(),  True),
    StructField("categoria",  StringType(),  True),
    StructField("precio",     DoubleType(),  True),
    StructField("cantidad",   IntegerType(), True),
    StructField("fecha",      StringType(),  True),
    StructField("valoracion", IntegerType(), True),
])

schema_productos = StructType([
    StructField("producto",   StringType(), True),
    StructField("categoria",  StringType(), True),
    StructField("proveedor",  StringType(), True),
    StructField("coste",      DoubleType(), True),
    StructField("stock",      IntegerType(),True),
])

# Creamos los DataFrames de Spark desde los documentos de MongoDB
df_ventas = spark.createDataFrame(
    [tuple(d[f.name] for f in schema_ventas) for d in ventas_list],
    schema=schema_ventas
)
df_productos = spark.createDataFrame(
    [tuple(d[f.name] for f in schema_productos) for d in productos_list],
    schema=schema_productos
)

print("DataFrame de ventas cargado desde MongoDB:")
df_ventas.show(5)
print(f"Total registros: {df_ventas.count()}")


# ──────────────────────────────────────────────
# 3. TRANSFORMACIONES CON SPARK
# ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("3. TRANSFORMACIONES CON SPARK (MapReduce y DataFrame)")
print("=" * 60)

# Añadimos columnas calculadas
# importe_total = precio * cantidad (el ingreso total del pedido)
df_ventas = df_ventas \
    .withColumn("importe_total", F.col("precio") * F.col("cantidad")) \
    .withColumn("fecha_date",    F.to_date(F.col("fecha"), "yyyy-MM-dd")) \
    .withColumn("mes",           F.month(F.col("fecha_date"))) \
    .withColumn("trimestre",     F.quarter(F.col("fecha_date")))

print("Columnas calculadas añadidas:")
df_ventas.select("id_pedido", "precio", "cantidad", "importe_total", "mes", "trimestre").show(5)

# ── MapReduce con RDD — conteo de pedidos por ciudad ──
print("MapReduce — pedidos por ciudad:")
rdd_ciudades = df_ventas.select("ciudad").rdd \
    .map(lambda row: (row[0], 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False)

for ciudad, count in rdd_ciudades.collect():
    barra = "█" * count
    print(f"  {ciudad:12s} {barra} ({count} pedidos)")

# ── JOIN entre ventas y productos ──
print("\nJOIN Spark: ventas + info de productos (margen de beneficio):")
df_join = df_ventas.join(
    df_productos.select("producto", "coste", "proveedor", "stock"),
    on="producto",
    how="inner"
)
# Calculamos el margen de beneficio por pedido
df_join = df_join.withColumn(
    "beneficio",
    (F.col("precio") - F.col("coste")) * F.col("cantidad")
)
df_join.select("id_pedido","producto","importe_total","coste","beneficio","proveedor").show(5)


# ──────────────────────────────────────────────
# 4. ANÁLISIS AVANZADO
# ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("4. ANÁLISIS AVANZADO CON SPARK")
print("=" * 60)

# Estadísticas por categoría
print("Estadísticas por categoría de producto:")
df_ventas.groupBy("categoria") \
    .agg(
        F.sum("importe_total").alias("ingresos_totales"),
        F.avg("precio").alias("precio_medio"),
        F.sum("cantidad").alias("unidades_vendidas"),
        F.avg("valoracion").alias("valoracion_media"),
        F.count("id_pedido").alias("num_pedidos")
    ) \
    .withColumn("ingresos_totales", F.round("ingresos_totales", 2)) \
    .withColumn("precio_medio",     F.round("precio_medio",     2)) \
    .withColumn("valoracion_media", F.round("valoracion_media", 2)) \
    .orderBy("ingresos_totales", ascending=False) \
    .show()

# Window Function — ranking de clientes por ciudad
print("Ranking de clientes por ciudad (Window Function):")
ventana = Window.partitionBy("ciudad").orderBy(F.col("importe_total").desc())
df_ranking = df_ventas \
    .groupBy("ciudad", "cliente") \
    .agg(F.round(F.sum("importe_total"), 2).alias("total_gastado")) \
    .withColumn("ranking", F.rank().over(Window.partitionBy("ciudad").orderBy(F.col("total_gastado").desc())))

df_ranking.filter(F.col("ranking") <= 2) \
    .orderBy("ciudad", "ranking") \
    .show()

# Evolución mensual de ventas
print("Evolución mensual de ingresos:")
df_mensual = df_ventas \
    .groupBy("mes") \
    .agg(F.round(F.sum("importe_total"), 2).alias("ingresos_mes")) \
    .orderBy("mes")
df_mensual.show()

# SQL directo sobre el DataFrame
df_ventas.createOrReplaceTempView("ventas")
df_productos.createOrReplaceTempView("productos")

print("Top 3 productos más rentables (SQL sobre Spark):")
spark.sql("""
    SELECT v.producto,
           v.categoria,
           ROUND(SUM(v.importe_total), 2)          AS ingresos,
           ROUND(SUM((v.precio - p.coste) * v.cantidad), 2) AS beneficio_total,
           ROUND(AVG(v.valoracion), 2)              AS valoracion_media
    FROM ventas v
    JOIN productos p ON v.producto = p.producto
    GROUP BY v.producto, v.categoria
    ORDER BY beneficio_total DESC
    LIMIT 3
""").show()


# ──────────────────────────────────────────────
# 5. ESCRIBIR RESULTADOS DE SPARK → MONGODB
# ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("5. SPARK → MONGODB (Escritura de resultados)")
print("=" * 60)

print("""
En producción el resultado se escribiría así:
  df_resultado.write.format("mongo")
    .option("uri", "mongodb://localhost/ecommerce.resumen_ventas")
    .mode("overwrite").save()

Aquí lo simulamos convirtiendo a pandas y escribiendo en mongomock.
""")

# Calculamos el resumen que queremos guardar en MongoDB
df_resumen = df_ventas \
    .groupBy("categoria", "mes") \
    .agg(
        F.round(F.sum("importe_total"), 2).alias("ingresos"),
        F.sum("cantidad").alias("unidades"),
        F.round(F.avg("valoracion"), 2).alias("valoracion_media"),
        F.count("id_pedido").alias("num_pedidos")
    ) \
    .orderBy("mes", "categoria")

# Convertimos a pandas para simular la escritura en MongoDB
resumen_pd = df_resumen.toPandas()

# Escribimos en MongoDB (colección nueva — resumen analítico)
resumen_docs = resumen_pd.to_dict(orient="records")
db["resumen_ventas"].insert_many(resumen_docs)

print(f"✔ {len(resumen_docs)} documentos de resumen escritos en MongoDB")
print("\nDocumentos guardados en MongoDB (colección resumen_ventas):")
for doc in db["resumen_ventas"].find({}, {"_id": 0}):
    print(f"  {doc}")


# ──────────────────────────────────────────────
# 6. VERIFICACIÓN EN MONGODB
# ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("6. VERIFICACIÓN — Aggregation Pipeline en MongoDB")
print("=" * 60)

print("""
Una vez que Spark ha escrito los resultados en MongoDB,
podemos usar el Aggregation Pipeline de MongoDB para
consultas rápidas sin necesidad de Spark.
Ideal para dashboards en tiempo real.
""")

# Consultamos el resumen desde MongoDB con Aggregation Pipeline
pipeline = [
    {"$group": {
        "_id":          "$categoria",
        "ingresos_total": {"$sum": "$ingresos"},
        "pedidos_total":  {"$sum": "$num_pedidos"},
    }},
    {"$project": {
        "categoria":      "$_id",
        "ingresos_total": {"$round": ["$ingresos_total", 2]},
        "pedidos_total":  1,
        "_id":            0
    }},
    {"$sort": {"ingresos_total": -1}}
]

print("Resumen final por categoría (desde MongoDB):")
for doc in db["resumen_ventas"].aggregate(pipeline):
    print(f"  {doc['categoria']:15s} → {doc['ingresos_total']:,.2f}€ "
          f"({doc['pedidos_total']} pedidos)")


# ──────────────────────────────────────────────
# 7. VISUALIZACIONES
# ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("7. VISUALIZACIONES")
print("=" * 60)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F9FA',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})
COLORS = ['#028090','#00A896','#02C39A','#F39C12','#C0392B','#2E86C1']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('PySpark + MongoDB — Análisis de Ventas E-commerce',
             fontsize=15, fontweight='bold')

# ── Gráfico 1: Ingresos por categoría
df_cat = df_ventas.groupBy("categoria") \
    .agg(F.round(F.sum("importe_total"), 2).alias("ingresos")) \
    .toPandas()
bars = axes[0,0].bar(df_cat['categoria'], df_cat['ingresos'],
                      color=COLORS[:len(df_cat)], edgecolor='white', linewidth=2)
axes[0,0].set_title('Ingresos totales por categoría', fontweight='bold')
axes[0,0].set_ylabel('Ingresos (€)')
for bar, val in zip(bars, df_cat['ingresos']):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                   f'{val:,.0f}€', ha='center', fontweight='bold', fontsize=9)

# ── Gráfico 2: Pedidos por ciudad (MapReduce)
ciudades_pd = pd.DataFrame(rdd_ciudades.collect(), columns=['ciudad','pedidos'])
bars2 = axes[0,1].barh(ciudades_pd['ciudad'], ciudades_pd['pedidos'],
                        color=COLORS[:len(ciudades_pd)], edgecolor='white', linewidth=1.5)
axes[0,1].set_title('Pedidos por ciudad (MapReduce)', fontweight='bold')
axes[0,1].set_xlabel('Nº pedidos')
for bar, val in zip(bars2, ciudades_pd['pedidos']):
    axes[0,1].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                   str(val), va='center', fontweight='bold', fontsize=10)

# ── Gráfico 3: Evolución mensual
df_mes_pd = df_mensual.toPandas()
axes[0,2].fill_between(df_mes_pd['mes'], df_mes_pd['ingresos_mes'], alpha=0.3, color=COLORS[0])
axes[0,2].plot(df_mes_pd['mes'], df_mes_pd['ingresos_mes'],
               marker='o', color=COLORS[0], linewidth=2.5, markersize=9)
axes[0,2].set_title('Evolución mensual de ingresos', fontweight='bold')
axes[0,2].set_xlabel('Mes')
axes[0,2].set_ylabel('Ingresos (€)')
axes[0,2].set_xticks(df_mes_pd['mes'])
axes[0,2].set_xticklabels([f'Mes {m}' for m in df_mes_pd['mes']])
for x, y in zip(df_mes_pd['mes'], df_mes_pd['ingresos_mes']):
    axes[0,2].text(x, y + 80, f'{y:,.0f}€', ha='center', fontsize=8, fontweight='bold')

# ── Gráfico 4: Beneficio por producto (JOIN Spark)
df_ben = df_join.groupBy("producto") \
    .agg(F.round(F.sum("beneficio"), 2).alias("beneficio_total")) \
    .orderBy("beneficio_total", ascending=True) \
    .toPandas()
bars3 = axes[1,0].barh(df_ben['producto'], df_ben['beneficio_total'],
                        color=COLORS, edgecolor='white', linewidth=1.5)
axes[1,0].set_title('Beneficio total por producto\n(JOIN Spark + MongoDB)', fontweight='bold')
axes[1,0].set_xlabel('Beneficio (€)')
for bar, val in zip(bars3, df_ben['beneficio_total']):
    axes[1,0].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                   f'{val:,.0f}€', va='center', fontsize=8, fontweight='bold')

# ── Gráfico 5: Valoración media por producto
df_val = df_ventas.groupBy("producto") \
    .agg(F.round(F.avg("valoracion"), 2).alias("valoracion_media")) \
    .orderBy("valoracion_media", ascending=False) \
    .toPandas()
colores_val = [COLORS[0] if v >= 4.5 else COLORS[3] if v >= 3.5 else COLORS[4]
               for v in df_val['valoracion_media']]
bars4 = axes[1,1].bar(range(len(df_val)), df_val['valoracion_media'],
                       color=colores_val, edgecolor='white', linewidth=1.5)
axes[1,1].set_title('Valoración media por producto', fontweight='bold')
axes[1,1].set_ylabel('Valoración (1-5)')
axes[1,1].set_ylim(0, 5.5)
axes[1,1].set_xticks(range(len(df_val)))
axes[1,1].set_xticklabels(df_val['producto'], rotation=30, ha='right', fontsize=8)
axes[1,1].axhline(y=4, color=COLORS[0], linestyle='--', linewidth=1.5, alpha=0.7, label='Objetivo ≥ 4')
axes[1,1].legend(fontsize=8)
for bar, val in zip(bars4, df_val['valoracion_media']):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                   f'{val}⭐', ha='center', fontsize=8, fontweight='bold')

# ── Gráfico 6: Flujo arquitectura PySpark + MongoDB
axes[1,2].axis('off')
axes[1,2].set_title('Arquitectura PySpark ↔ MongoDB', fontweight='bold')

bloques = [
    (0.05, 0.65, 0.2, 0.25, '#566573', 'Fuentes\nOperacionales\n(APIs, IoT, Apps)'),
    (0.32, 0.60, 0.2, 0.35, '#028090', 'MongoDB\n(Data Lake /\nOperacional)'),
    (0.59, 0.60, 0.2, 0.35, '#1A1A2E', 'PySpark\n(Procesamiento\nDistribuido)'),
    (0.82, 0.65, 0.16, 0.25, '#00A896', 'MongoDB\n(Resultados\nAnalíticos)'),
]
for (x, y, w, h, color, label) in bloques:
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor='white',
                          linewidth=2, transform=axes[1,2].transAxes, zorder=3)
    axes[1,2].add_patch(rect)
    axes[1,2].text(x + w/2, y + h/2, label, transform=axes[1,2].transAxes,
                   ha='center', va='center', fontsize=8, color='white',
                   fontweight='bold', zorder=4)

# Flechas entre bloques
for start_x, end_x, y_pos in [(0.25, 0.32, 0.77), (0.52, 0.59, 0.77), (0.79, 0.82, 0.77)]:
    axes[1,2].annotate('', xy=(end_x, y_pos), xytext=(start_x, y_pos),
                        xycoords='axes fraction', textcoords='axes fraction',
                        arrowprops=dict(arrowstyle='->', color='#566573', lw=2))

# Etiquetas de las flechas
axes[1,2].text(0.285, 0.82, 'insert', transform=axes[1,2].transAxes,
               ha='center', fontsize=7, color='#566573', style='italic')
axes[1,2].text(0.555, 0.82, 'read', transform=axes[1,2].transAxes,
               ha='center', fontsize=7, color='#566573', style='italic')
axes[1,2].text(0.805, 0.82, 'write', transform=axes[1,2].transAxes,
               ha='center', fontsize=7, color='#566573', style='italic')

# Nota de producción
axes[1,2].text(0.5, 0.45,
    'En producción:\nspark.read.format("mongo")\n  .option("uri","mongodb://...")\n  .load()',
    transform=axes[1,2].transAxes, ha='center', fontsize=8,
    color='#1A1A2E', style='italic',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#E8F8F5', edgecolor='#028090'))

plt.tight_layout()
plt.savefig('pyspark_mongodb.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Notebook completo")
print("\nFlujo visto en este ejemplo:")
print("  1. MongoDB    → almacena documentos operacionales")
print("  2. Spark      → lee y convierte documentos a DataFrame")
print("  3. MapReduce  → conteo de pedidos por ciudad con RDD")
print("  4. JOIN       → ventas + productos para calcular beneficio")
print("  5. Window Fn  → ranking de clientes por ciudad")
print("  6. SQL        → consultas analíticas sobre DataFrames")
print("  7. MongoDB    → escritura de resultados analíticos")
print("  8. Pipeline   → consulta final desde MongoDB sin Spark")

spark.stop()